# 01: Understanding Face Embeddings

## What are Face Embeddings?

A **face embedding** is a numerical representation of a face - a list of numbers that captures the unique characteristics of a person's face.

Instead of storing an image (millions of pixels), we store **512 numbers** that represent the essence of that face. This is incredibly powerful for face recognition!

### Key Concepts:

1. **High-dimensional Vector**: 512 dimensions
2. **Learned Representation**: Generated by deep neural networks (ArcFace)
3. **Similarity-based**: Similar faces have similar embeddings
4. **Efficient**: Fast to compare and search

---

## Why Embeddings?

### Problem: How do computers recognize faces?

**Naive approach**: Compare images pixel by pixel
- ❌ Two photos of same person look different (angle, lighting, expression)
- ❌ Comparing millions of pixels is slow
- ❌ No understanding of facial features

**Smart approach**: Use face embeddings
- ✅ Same person ≈ similar embeddings
- ✅ Just 512 numbers instead of millions of pixels
- ✅ Fast mathematical comparison

### How it works:

```
Photo of John → Deep Neural Network (ArcFace) → Embedding (512 numbers)
Photo of John → Deep Neural Network (ArcFace) → Similar Embedding
Photo of Sarah → Deep Neural Network (ArcFace) → Different Embedding
```

## The Magic: How ArcFace Works

### ArcFace: Additive Angular Margin

ArcFace is a deep learning model that learns to generate face embeddings by:

1. **Training on millions of face images**
2. **Learning to maximize distance** between different people
3. **Learning to minimize distance** between same person
4. **Using angular margins** for better separation

### The Embedding Space:

Imagine a 512-dimensional space:
- Each face is a point in this space
- Same person's faces cluster together
- Different people's faces are far apart
- We can find matching faces by finding the closest point!

In [ ]:
import os
import sys
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from cv_pipeline import FaceRecognizer, FaceDetector

print("✅ Imports successful!")

## Generating Embeddings

Let's generate face embeddings from images and explore their properties:

In [ ]:
# Initialize the recognizer
print("Loading ArcFace model...")
recognizer = FaceRecognizer()
detector = FaceDetector()
print("✅ Models loaded!")

### Try with Your Own Image

Upload a face image and generate its embedding:

In [ ]:
# Example: Load and process an image
# Replace with your own image path

# For this example, let's create a simple test
# You should replace this with actual face images

# image_path = "path/to/your/face/image.jpg"
# image = cv2.imread(image_path)
# 
# if image is None:
#     print("❌ Could not load image. Please check the path.")
# else:
#     # Detect face
#     detections = detector.detect(image)
#     print(f"Found {len(detections)} face(s)")
#     
#     # Generate embedding
#     embedding = recognizer.get_embedding(image)
#     print(f"\n✅ Embedding generated!")
#     print(f"Shape: {embedding.shape}")
#     print(f"Type: {embedding.dtype}")
#     print(f"\nFirst 20 values:")
#     print(embedding[:20])
#     print(f"\nEmbedding statistics:")
#     print(f"Min: {embedding.min():.4f}")
#     print(f"Max: {embedding.max():.4f}")
#     print(f"Mean: {embedding.mean():.4f}")
#     print(f"Std: {embedding.std():.4f}")

print("To generate an embedding, follow the steps above with your own face image.")

## Visualizing Embeddings

Since embeddings are 512-dimensional, we can't visualize them directly.
Instead, we use **dimensionality reduction** to project them to 2D:

In [ ]:
def visualize_embeddings_2d(embeddings, labels, title="Embedding Visualization"):
    """
    Visualize embeddings in 2D using PCA
    
    LEARNING: PCA (Principal Component Analysis) reduces dimensions
    while preserving the most important variance.
    """
    if len(embeddings) < 2:
        print("Need at least 2 embeddings to visualize")
        return
    
    # Reduce to 2D using PCA
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)
    
    print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")
    print(f"  Component 1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  Component 2: {pca.explained_variance_ratio_[1]:.2%}")
    
    # Create visualization
    plt.figure(figsize=(10, 8))
    
    # Get unique labels
    unique_labels = list(set(labels))
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    
    for i, label in enumerate(unique_labels):
        mask = np.array(labels) == label
        plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   label=label, color=colors[i], s=100, alpha=0.6, edgecolors='k')
    
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("✅ Visualization function defined!")
print("Use this to visualize embeddings from multiple face images.")

## Similarity Matching

### Comparing Embeddings

We use **cosine similarity** to compare two embeddings:

- **Formula**: `similarity = (v1 · v2) / (||v1|| × ||v2||)`
- **Range**: 0 to 1
- **Interpretation**:
  - `> 0.6` = likely same person ✅
  - `0.4-0.6` = maybe same person ⚠️
  - `< 0.4` = different person ❌

In [ ]:
def compare_embeddings_example(embedding1_2d, embedding2_2d):
    """
    Example of comparing two embeddings
    """
    # Cosine similarity
    similarity = cosine_similarity([embedding1_2d], [embedding2_2d])[0][0]
    
    print(f"Similarity score: {similarity:.4f}")
    
    if similarity > 0.6:
        print(f"✅ MATCH! Confidence: {similarity:.2%}")
    elif similarity > 0.4:
        print(f"⚠️  MAYBE - Need more samples for confirmation")
    else:
        print(f"❌ NO MATCH - Different people")
    
    return similarity

print("✅ Comparison function defined!")

## Key Takeaways

1. **Face embeddings** are 512-dimensional vectors that represent faces
2. **ArcFace** is trained to make same-person faces close and different-person faces far apart
3. **Cosine similarity** measures how similar two embeddings are (0 to 1)
4. **Fast matching**: Instead of comparing millions of pixels, we compare 512 numbers!
5. **Scalable**: With pgvector, we can search millions of faces in milliseconds

## Next Steps

- ➡️ Check `02_pgvector_tutorial.ipynb` to learn how to store and search embeddings in a database
- ➡️ Try the Streamlit app for real-world face recognition
- ➡️ Experiment with different similarity thresholds